In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
!wget https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip

--2026-04-08 13:05:07--  https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘human+activity+recognition+using+smartphones.zip’

human+activity+reco     [        <=>         ]  58.18M  36.3MB/s    in 1.6s    

2026-04-08 13:05:09 (36.3 MB/s) - ‘human+activity+recognition+using+smartphones.zip’ saved [61005872]



In [ ]:
!unzip human+activity+recognition+using+smartphones.zip

Archive:  human+activity+recognition+using+smartphones.zip
 extracting: UCI HAR Dataset.names   
 extracting: UCI HAR Dataset.zip     


In [ ]:
!unzip "UCI HAR Dataset.zip"

Archive:  UCI HAR Dataset.zip
   creating: UCI HAR Dataset/
  inflating: UCI HAR Dataset/.DS_Store  
   creating: __MACOSX/
   creating: __MACOSX/UCI HAR Dataset/
  inflating: __MACOSX/UCI HAR Dataset/._.DS_Store  
  inflating: UCI HAR Dataset/activity_labels.txt  
  inflating: __MACOSX/UCI HAR Dataset/._activity_labels.txt  
  inflating: UCI HAR Dataset/features.txt  
  inflating: __MACOSX/UCI HAR Dataset/._features.txt  
  inflating: UCI HAR Dataset/features_info.txt  
  inflating: __MACOSX/UCI HAR Dataset/._features_info.txt  
  inflating: UCI HAR Dataset/README.txt  
  inflating: __MACOSX/UCI HAR Dataset/._README.txt  
   creating: UCI HAR Dataset/test/
   creating: UCI HAR Dataset/test/Inertial Signals/
  inflating: UCI HAR Dataset/test/Inertial Signals/body_acc_x_test.txt  
   creating: __MACOSX/UCI HAR Dataset/test/
   creating: __MACOSX/UCI HAR Dataset/test/Inertial Signals/
  inflating: __MACOSX/UCI HAR Dataset/test/Inertial Signals/._body_acc_x_test.txt  
  inflating: UCI HAR

In [ ]:
import os

print(os.listdir())
print(os.listdir('UCI HAR Dataset'))
print(os.listdir("UCI HAR Dataset/train"))
print(os.listdir("UCI HAR Dataset/train/Inertial Signals"))

['.config', 'UCI HAR Dataset', 'UCI HAR Dataset.names', 'human+activity+recognition+using+smartphones.zip', '__MACOSX', 'UCI HAR Dataset.zip', 'sample_data']
['activity_labels.txt', '.DS_Store', 'train', 'README.txt', 'test', 'features_info.txt', 'features.txt']
['Inertial Signals', 'subject_train.txt', 'X_train.txt', 'y_train.txt']
['body_gyro_x_train.txt', 'body_gyro_y_train.txt', 'body_acc_y_train.txt', 'body_acc_x_train.txt', 'total_acc_y_train.txt', 'total_acc_z_train.txt', 'body_acc_z_train.txt', 'total_acc_x_train.txt', 'body_gyro_z_train.txt']


In [19]:
DATASET_PATH = "/content/UCI HAR Dataset/"

def load_signal(filepath):
    return pd.read_csv(filepath, header=None, delim_whitespace=True).values.astype('float32')

def load_raw_signal(split):
  path = DATASET_PATH + split + "/Inertial Signals/"
  signals = [
        'body_acc_x', 'body_acc_y', 'body_acc_z',
        'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
        'total_acc_x', 'total_acc_y', 'total_acc_z'
    ]
  loaded = [load_signal(f"{path}{s}_{split}.txt") for s in signals]
    # Each loaded[i] is (samples, 128) → stack → (samples, 128, 9)
  return np.dstack(loaded)

X_train = load_raw_signal('train')
X_test = load_raw_signal('test')
print(f"Shape of X_train {X_train.shape}")
print(f"Shape of X_test {X_test.shape}")

/tmp/ipykernel_23909/2776727782.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values.astype('float32')
/tmp/ipykernel_23909/2776727782.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values.astype('float32')
/tmp/ipykernel_23909/2776727782.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  return pd.read_csv(filepath, header=None, delim_whitespace=True).values.astype('float32')
/tmp/ipykernel_23909/2776727782.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  re

Shape of X_train (7352, 128, 9)
Shape of X_test (2947, 128, 9)


In [18]:
print(f"X_train type: {X_train.dtype}") # Should be float32, NOT object


X_train type: object


In [11]:
def load_labels(split):
    path = DATASET_PATH + split + f'/y_{split}.txt'
    return pd.read_csv(path, header=None).values.flatten() - 1  # 0-indexed

y_train = load_labels('train')   # values: 0–5
y_test  = load_labels('test')


In [12]:
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes=6)
y_test_cat  = to_categorical(y_test,  num_classes=6)


In [20]:
# If you are using categorical_crossentropy, ensure labels are one-hot encoded floats
y_train_cat = y_train_cat.astype('float32')
y_test_cat = y_test_cat.astype('float32')

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()
model.add(Bidirectional(LSTM(64,return_sequences=True),input_shape=(128,9)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.5))
model.add(Dense(6, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Early = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights = True
)

model.fit(X_train, y_train_cat,
          epochs=20, batch_size=32,
          validation_data=(X_test, y_test_cat),
          callbacks = [Early])

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_8 (Bidirectional) │ (None, 128, 128)       │        37,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,494 (310.52 KB)

 Trainable params: 79,494 (310.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 55s 212ms/step - accuracy: 0.6250 - loss: 0.9032 - val_accuracy: 0.7397 - val_loss: 0.6322
Epoch 2/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 46s 202ms/step - accuracy: 0.7908 - loss: 0.5223 - val_accuracy: 0.8331 - val_loss: 0.5217
Epoch 3/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 82s 202ms/step - accuracy: 0.8836 - loss: 0.3335 - val_accuracy: 0.8229 - val_loss: 0.5164
Epoch 4/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 82s 202ms/step - accuracy: 0.9136 - loss: 0.2561 - val_accuracy: 0.8724 - val_loss: 0.4102
Epoch 5/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 46s 202ms/step - accuracy: 0.9329 - loss: 0.1843 - val_accuracy: 0.8741 - val_loss: 0.4083
Epoch 6/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 47s 204ms/step - accuracy: 0.9276 - loss: 0.2067 - val_accuracy: 0.8894 - val_loss: 0.3930
Epoch 7/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 81s 198ms/step - accuracy: 0.9378 - loss: 0.1666 - val_accuracy: 0.8907 - val_loss: 0.3424
Epoch 8/20
230/230 ━━━━━━━━━━━━━━━━━━━━ 47s 203ms/step - accuracy: 0.9425 - loss: 0

In [23]:
# Save the model
model.save('har_model.keras')

# If you are in Google Colab, download it to your computer:
from google.colab import files
files.download('har_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import tensorflow as tf

# Convert the model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Add this line to handle the Bidirectional LSTM ops specifically
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
tflite_model = converter.convert()

# Save the .tflite file
with open('har_model.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpfc2g9ekz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 9), dtype=tf.float32, name='keras_tensor_55')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  133638928166032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928165648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928167376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928167568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928166416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928167952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928167184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928166224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928169488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928169680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133638928168528:

In [25]:
from google.colab import files

# This will trigger a browser download prompt
files.download('har_model.tflite')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>